In [1]:
!pip install pdfplumber openpyxl pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 114.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter


# ----------------------------------------------------------------------
# 1. Extração do texto do PDF
# ----------------------------------------------------------------------
def extrair_texto(caminho_pdf: str) -> str:
    """Extrai e concatena o texto de todas as páginas do PDF."""
    with pdfplumber.open(caminho_pdf) as pdf:
        paginas = [pagina.extract_text() or "" for pagina in pdf.pages]
    return "\n".join(paginas)


# ----------------------------------------------------------------------
# 2. Parsing do texto para uma lista de registros
# ----------------------------------------------------------------------
CODIGOS_SITUACAO = {"024", "100", "115", "120", "135", "200", "250", "300", "301", "999"}

NOISE_PATTERNS = [
    re.compile(r"^Pag\.:\s*\d+$"),
    re.compile(r"^Apuração Colaborador$"),
    re.compile(r"^Período de:"),
    re.compile(r"^HRAP110\.APU"),
]

HEADER_RE = re.compile(r"^(\d{5})\s+(.+?)\s+(\d{4})$")
DATE_RE = re.compile(r"^(\d{2}/\d{2}/\d{2})\s+(\w{3})\s+(.*)$")
TOTAL_HORAS_RE = re.compile(r"^\d{3,}:\d{2}$")


def _is_noise(linha: str) -> bool:
    return any(p.match(linha) for p in NOISE_PATTERNS)


def parse_relatorio(texto: str) -> list[dict]:
    linhas = [l.strip() for l in texto.split("\n") if l.strip()]

    registros = []
    matricula_atual = nome_atual = None
    em_totais = False
    ultimo_dia = ultima_marcacao = ""

    for linha in linhas:
        if _is_noise(linha):
            continue

        m = HEADER_RE.match(linha)
        if m:
            matricula, nome, _codigo_turno = m.groups()
            if matricula != matricula_atual:
                matricula_atual, nome_atual = matricula, nome
                ultimo_dia = ultima_marcacao = ""
            em_totais = False
            continue

        if linha == "Dia Marcações Situações Apuradas Horas":
            continue

        if linha.startswith("Total Colaborador:"):
            em_totais = True
            continue

        tokens = linha.split()
        if tokens and TOTAL_HORAS_RE.match(tokens[-1]):
            continue
        if em_totais:
            continue

        dm = DATE_RE.match(linha)
        tem_data_propria = bool(dm)
        if dm:
            data_bruta, _dow, resto = dm.groups()
            d, mo, y = data_bruta.split("/")
            dia = f"{d}/{mo}/20{y}"
            ultimo_dia = dia
        else:
            resto = linha
            dia = ultimo_dia

        tokens = resto.split()
        idx_codigo = next((i for i, t in enumerate(tokens) if t in CODIGOS_SITUACAO), None)
        if idx_codigo is None:
            continue

        marc_tokens = tokens[:idx_codigo]
        codigo = tokens[idx_codigo]
        desc_tokens = tokens[idx_codigo + 1 : -1]
        horas = tokens[-1]

        if marc_tokens:
            marcacoes = " ".join(marc_tokens)
            ultima_marcacao = marcacoes
        elif tem_data_propria:
            marcacoes = ""
            ultima_marcacao = ""
        else:
            marcacoes = ultima_marcacao

        registros.append(
            {
                "MATRICULA": int(matricula_atual),
                "EMPREGADO": nome_atual,
                "DIA": dia,
                "MARCAÇÕES": marcacoes,
                "SITUAÇÕES APURADAS": f"{codigo} {' '.join(desc_tokens)}",
                "HORAS": horas,
            }
        )

    return registros


# ----------------------------------------------------------------------
# 3. Geração do Excel
# ----------------------------------------------------------------------
def gerar_excel(registros: list[dict], caminho_saida: str) -> None:
    df = pd.DataFrame(registros, columns=[
        "MATRICULA", "EMPREGADO", "DIA", "MARCAÇÕES", "SITUAÇÕES APURADAS", "HORAS"
    ])

    with pd.ExcelWriter(caminho_saida, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Apuracao")
        ws = writer.sheets["Apuracao"]

        fonte_cabecalho = Font(name="Arial", bold=True, color="FFFFFF")
        preenchimento = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
        for cel in ws[1]:
            cel.font = fonte_cabecalho
            cel.fill = preenchimento
            cel.alignment = Alignment(horizontal="center")

        for linha in ws.iter_rows(min_row=2, max_row=ws.max_row):
            for cel in linha:
                cel.font = Font(name="Arial")

        larguras = {"A": 12, "B": 38, "C": 13, "D": 22, "E": 34, "F": 10}
        for col, largura in larguras.items():
            ws.column_dimensions[col].width = largura

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"


# ----------------------------------------------------------------------
# 4. Uso (isso substitui o bloco de linha de comando / argparse)
# ----------------------------------------------------------------------
texto = extrair_texto("HRAP110_TMPFD40.PDF")
registros = parse_relatorio(texto)
gerar_excel(registros, "saida.xlsx")

print(f"{len(registros)} linhas geradas para {len({r['MATRICULA'] for r in registros})} colaboradores")


1798 linhas geradas para 573 colaboradores
